In [1]:
"""
Class 26. Experiment Tracking with MLFlow

Basic rules for experiment:
1. Always use pen and paper to draft your thoughts and plans

1.1. How an example data is processed for training?
     split the data into train, test, val sets
            seed = 42, seed = 18, 
     get_ready_with_ai(file_path):
         task 1. I will load the image using the filepath
         task 2. I will reshape the image to fit the model (64x64)
         task 3. I will convert the images into a fixed color model
         task 4. I will normalize the pixel values
         
1.2. Design the architecture of the model
     Input shape = B x 64 x 64
     Layer 1: output shape = B x 32 x 32
     Layer 2: output shape = B x 16 x 16
     Layer 3: output shape = B x 8 x 8
     Layer 4: output shape = B x 16
     Layer 5: output shape = B x 10
     
1.3. Training details:
      Batch size, epoch, learning rate, learning rate scheduler, optimizer, loss function, metrics
      
Objectives:
1. Build a machine learning training pipeline using PyTorch Lightning
2. Track experiment using MLFlow

Installation: pip install mlflow pytorch_lightning
"""


import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import pandas as pd
import os
import torch.nn as nn
import pytorch_lightning as pl
from mlflow.models import infer_signature
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import mlflow

mlflow.set_experiment(
    experiment_name='hand-written-digit-recognition-using-cnn'
)

<Experiment: artifact_location='file:///E:/PyCharmProjects/BongoDev/notebooks/mlruns/1', creation_time=1778340073798, experiment_id='1', last_update_time=1778340073798, lifecycle_stage='active', name='hand-written-digit-recognition-using-cnn', tags={}, trace_location=None, workspace='default'>

In [2]:
"""Files and directories """
ROOT_DIR = "E:\\PyCharmProjects\\pythonProject\\"
DATA_DIR = os.path.join(ROOT_DIR, "data")
dataset_file = os.path.join(DATA_DIR, "digit_train.csv")

ARTIFACT_FOLDER_NAME = 'model'
# Path of the source code
SOURCE_CODE_PATH = os.path.join(
    os.getcwd(), 'class-18-experiment-tracking-with-mlflow.ipynb'
)
# Name of the source code file in the saved dir
SOURCE_CODE_ARTIFACT = 'trainer.ipynb'

In [3]:
""" Hyperparameters 
Parameters that are not for neural networks but use to train
models.
"""
EPOCHS = 3
BATCH_SIZE = 64
LEARNING_RATE = 0.0001
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15

In [4]:
""" Seed for reproducibility """

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


### Data Pipeline

In [5]:
class MyDataset(Dataset):
    def __init__(self, file_path, transform):
        self.data = pd.read_csv(file_path)
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        """ Get a sample from the dataset. """
        example = self.data.iloc[idx]
        pixels = example.values[1:].astype('float32')
        pixels /= 255.0
        label = int(example.values[0])
        
        """
        pixels: B x C x H x W
        """
        pixels = torch.tensor(pixels).reshape(28, 28).unsqueeze(0)
        label = torch.tensor(label)
        
        if self.transform:
            pixels = self.transform(pixels)
        
        return pixels, label

In [6]:
dataset = MyDataset(
    file_path=dataset_file,
    transform=transforms.Compose([
            transforms.Normalize(
                mean=torch.Tensor([0.1307]), 
                std=torch.Tensor([0.3081])
        )
    ])
)

In [7]:
train_size = int(TRAIN_SPLIT * len(dataset))
val_size = int(VAL_SPLIT * len(dataset))
test_size = len(dataset) - train_size - val_size
print(train_size, val_size, test_size)
train_dataset, val_dataset, test_dataset = random_split(
    dataset=dataset, 
    lengths=[train_size, val_size, test_size]
)
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

29399 6300 6301
29399
6300
6301


In [9]:
"""
How data loader works?

Step 1: n = dataset.__len__() 
Step 2: idx = [0, 1, 2, ..., n-1]
Step 3: Shuffle = True, then idx = [5, 1, 2, 8, 0, ...n-1, 10]
Step 4: Make the batches of ids from the idx list i.e. batches = [b1, b2, b3, ...]
Step 5: To load a single batch[i]:
        5.1. for each example eid in batch[i]:
                 pixels, label = dataset.__getitem__(eid)
                 processed_pixels.add(pixels)
                 processed_labels.add(label)
"""

for pixels, labels in train_loader:
    print(pixels.shape)
    print(labels.shape)
    break

torch.Size([64, 1, 28, 28])
torch.Size([64])


### Model

In [10]:
class DigitClassifier(pl.LightningModule):
    def __init__(self):
        super(DigitClassifier, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        return x
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=LEARNING_RATE)
        return optimizer
    
    def training_step(self, batch, batch_idx):
        pixels, labels = batch
        pixels, labels = pixels.to(device), labels.to(device)
        output = self.forward(pixels)
        loss = self.loss_fn(output, labels)
        self.log('train_loss', loss)
        return loss # You must return the value that need to be optimized
    
    def validation_step(self, batch, batch_idx):
        pixels, labels = batch
        pixels = pixels.to(device)
        labels = labels.to(device)
        output = self.forward(pixels)
        loss = self.loss_fn(output, labels)
        acc = (output.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', acc, prog_bar=True)
    
    def test_step(self, batch, batch_idx):
        pixels, labels = batch
        pixels = pixels.to(device)
        labels = labels.to(device)
        output = self.forward(pixels)
        loss = self.loss_fn(output, labels)
        acc = (output.argmax(dim=1) == labels).float().mean()
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_accuracy', acc, prog_bar=True)

In [13]:
early_stopping = EarlyStopping(monitor='val_loss', patience=2, verbose=True)
checkpoint_callback = ModelCheckpoint(monitor='val_accuracy', save_top_k=1, mode='max')

In [14]:
checkpoint_dir = os.path.join(os.getcwd(), "checkpoints")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
    
checkpoint_path = os.path.join(checkpoint_dir, "best_model.pth")

In [23]:
"""Experiment Run"""

with mlflow.start_run(run_name='feature-engineering-using-basic-pipeline') as run: 
    # hyperparameters
    mlflow.log_param('lr', LEARNING_RATE)
    mlflow.log_param('batch_size', BATCH_SIZE)
    mlflow.log_param('epochs', EPOCHS)
    
    model = DigitClassifier()
    
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
    )
    
    train_loader = DataLoader(
        dataset=train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True
    )
    val_loader = DataLoader(
        dataset=val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False
    )
    test_loader = DataLoader(
        dataset=test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False
    )
    
    trainer.fit(
        model=model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )
    # Evaluation result
    score = trainer.test(
        model=model,
        dataloaders=test_loader
    )
    print(score)
    
    # Log evaluation metrics
    test_acc = score[0]['test_accuracy']
    test_loss = score[0]['test_loss']
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_loss", test_loss)
    
    # Save the model
    pixels_batch = next(iter(test_loader))[0]
    pixels_batch = pixels_batch.cpu().numpy()
    signature = infer_signature(model, pixels_batch)
    
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path=ARTIFACT_FOLDER_NAME,
        input_example=pixels_batch,
        signature=signature
    )
    # Save the source code
    import shutil
    shutil.copyfile(SOURCE_CODE_PATH, SOURCE_CODE_ARTIFACT)
    mlflow.log_artifact(SOURCE_CODE_ARTIFACT)
    

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | loss_fn | CrossEntropyLoss | 0      | train | 0    
1 | fc1     | Linear           | 100 K  | train | 0    
2 | fc2     | Linear           | 8.3 K  | train | 0    
3 | fc3     | Linear           | 650    | train | 0    
-------------------------------------------------------------
109 K     Trainable params
0         Non-trainable params
109 K     Total params
0.43

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Testing: |          | 0/? [00:00<?, ?it/s]

2026/05/09 22:07:57 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/05/09 22:07:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9176321029663086
        test_loss           0.2794911861419678
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
[{'test_loss': 0.2794911861419678, 'test_accuracy': 0.9176321029663086}]


2026/05/09 22:07:58 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/05/09 22:08:30 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: setting an array element with a sequence.


In [24]:
print(f"mlflow ui --backend-store-uri {mlflow.get_tracking_uri()}")

mlflow ui --backend-store-uri sqlite:///E:/PyCharmProjects/BongoDev/notebooks/mlflow.db
